# Part A — Histogram Matching preprocessing pipeline

A classical histogram-matching preprocessing pipeline for cervical
colposcopy images. This notebook **removes** every component of the
former QI-SABP pipeline (saliency fusion, soft/hard masks, bit
allocation, dither, NLM denoise, bilateral post-filter, glare mask,
whiteness channel, rim rule, YCbCr 4:2:0 subsampling). The only
preprocessing operator applied between the raw frame and the validator
is **histogram matching** to a dataset-derived reference.

## Why histogram matching for colposcopy

Cervical colposcopy sequences are acquired while the speculum position,
the camera gain and the illuminator intensity all drift, so consecutive
frames of the *same* patient have markedly different luma and chroma
distributions. The diagnostic signal — the acetowhite response of the
transformation zone after acetic-acid application — is a *relative*
colour change, not an absolute one. Any per-frame absolute-colour
descriptor (mean L*, mean a*, saturation histogram) is therefore
contaminated by illumination drift and cannot be compared across frames
or against an atlas. Histogram matching re-maps each frame's per-channel
cumulative distribution onto a fixed reference CDF, so that frames
acquired under different illumination land on a common tone scale before
any colour or structure metric is computed.

The reference is the **per-channel median image** of the in-FOV pixels
across the whole patient sequence (computed once, then frozen). The
median (rather than the mean) is used because colposcopy frames contain
specular-glare dots and occasional saturation spikes that would corrupt
a mean reference but are suppressed by a per-pixel median. The reference
CDF is therefore the *typical* tone curve of this patient under typical
illumination, which is exactly the curve we want every frame to adopt.

## Pipeline overview

* Step 1 — Unsharp-mask sharpen (carried over from Step 1, shared with
  the rest of the suite; `SHARPEN_AMOUNT = 0.8`, `SHARPEN_KSIZE = 5`).
* Step 2 — Field-of-view (FOV) mask via grey-threshold + largest
  connected component + convex-hull fill. Used only to confine the
  histogram and the matching to tissue pixels, never to the black
  periphery.
* Step 3 — Build the **reference CDF** from the per-channel median of
  in-FOV pixels across the sharpened stack. Frozen for the run.
* Step 4 — **Histogram matching** (skimage `match_histograms`, multi-
  channel, on the RGB stack) of each frame to the reference, then
  optional linear blend `out = (1-strength)*orig + strength*matched`
  with `strength = HM_STRENGTH` (default 1.0 = full match).
* Step 5 — FOV pass-through: pixels outside the FOV are forced back to
  zero so the validator's FOV-masked metrics are computed on tissue
  only, exactly as in the rest of the suite.

## What this notebook does NOT claim

* It does NOT produce a compressed byte stream. The output PNG is the
  same size as the input PNG.
* It does NOT compare techniques against each other (that requires the
  multi-patient / multi-technique notebook).
* It does NOT demonstrate a clinical benefit on this single patient.
* It does NOT carry over any "quantum-inspired" allocation or
  information-bounded bit budget — those concepts belonged to QI-SABP
  and have been removed. The `Luma bits` metric in Part B is therefore
  reinterpreted as the **effective Shannon entropy of the in-FOV luma
  channel** (bits/pixel), not an analytic bit allocation.


In [1]:
# ============================================================
# Cell 1.1 -- Combined imports & global setup
# ============================================================
import os, sys, glob, time, json, math, random, warnings, platform
from collections import defaultdict
from pathlib import Path

import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt

import pandas as pd
from scipy import ndimage, stats
from scipy.fft import fft2, fftshift
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr, spearmanr
try:
    from skimage.exposure import match_histograms as _sk_match_hist
    _SKIMAGE_OK = True
except ImportError:
    _SKIMAGE_OK = False
    print("[warn] scikit-image not available — histogram matching will fail.")
try:
    from skimage.metrics import structural_similarity as _ssim_fn
    from skimage.color import deltaE_cie76 as _deltaE_fn, rgb2lab as _rgb2lab_fn
    _SKIMAGE_METRICS_OK = True
except ImportError:
    _SKIMAGE_METRICS_OK = False
    print("[warn] scikit-image metrics not available — validation will fail.")

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Output directory for results / CSV / JSON artefacts
SAVE_DIR = 'results'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Histogram-Matching Pipeline — Python {platform.python_version()}")
print(f"NumPy {np.__version__} | Pandas {pd.__version__} | OpenCV {cv2.__version__}")
print(f"Random seed: {SEED}")
print(f"Output directory: {SAVE_DIR}")
print("All imports OK.")


Histogram-Matching Pipeline — Python 3.10.20
NumPy 2.2.6 | Pandas 2.3.3 | OpenCV 4.11.0
Random seed: 42
Output directory: results
All imports OK.


In [2]:
# ============================================================
# Cell 2 -- Configuration
#   Single-class invasive carcinoma dataset.
#   Expected layout (configurable below):
#       <DATASET_ROOT>/<INVASIVE_DIR>/images/*.jpg
#       <DATASET_ROOT>/<INVASIVE_DIR>/<MASKS_SUBDIR>/*.png
# ============================================================

# ---- Image size ----
IMG_SIZE = 200

# ---- Data ----
DATASET_ROOT   = "C:/CER/dataset"   # <-- your dataset root
INVASIVE_DIR   = "invasive cancer"  # <-- single class folder (note the space)
MASKS_SUBDIR   = "masks"            # set "" if masks live directly under invasive cancer/
CLASSES        = ["invasive cancer"]

# ---- Sharpen (Step 1, shared across the suite) ----
APPLY_SHARPEN  = True
SHARPEN_AMOUNT = 0.8
SHARPEN_KSIZE  = 5

# ---- Field-of-view mask (used only to confine HM to tissue) ----
FOV_GRAY_THR   = 15        # pixels darker than this are "outside"
FOV_OPEN_KSIZE = 5         # morphological opening kernel
FOV_CLOSE_KSIZE = 11       # convex-hull-edge closing kernel

# ---- Histogram Matching (Step 2 -- the preprocessing operator) ----
HM_REFERENCE       = "median"   # "median" | "mean" | "first"
                                   # how to build the reference image from the stack
HM_MATCH_SPACE     = "RGB"      # "RGB" (match each of R,G,B independently)
HM_STRENGTH        = 1.0        # 1.0 = full match, 0.0 = identity,
                                   # 0..1 = linear blend (out = (1-s)*orig + s*matched)
HM_FOV_ONLY        = True       # only match inside FOV; force outside to zero
HM_CLIP            = True       # clip output to [0, 255]

# ---- Validation batch ----
N_PER_CLASS_VALIDATE = 5
SUBSET_PER_CLASS     = None

# Global flag set later by the loader -- declared here so downstream
# cells can reference it without NameError.
SYNTHETIC_MODE = False

print(f"Configuration OK  |  classes={CLASSES}  |  img={IMG_SIZE}x{IMG_SIZE}")
print(f"  sharpen={APPLY_SHARPEN} (amount={SHARPEN_AMOUNT}, ksize={SHARPEN_KSIZE})")
print(f"  FOV: gray_thr={FOV_GRAY_THR}, open={FOV_OPEN_KSIZE}, close={FOV_CLOSE_KSIZE}")
print(f"  histogram matching:")
print(f"    reference     = {HM_REFERENCE}")
print(f"    match_space   = {HM_MATCH_SPACE}")
print(f"    strength      = {HM_STRENGTH}  (1.0 = full match, 0.0 = identity)")
print(f"    fov_only      = {HM_FOV_ONLY}")
print(f"    clip          = {HM_CLIP}")
print(f"  validation batch = {N_PER_CLASS_VALIDATE} imgs/class")
print(f"  NOTE: HM does NOT quantise, so 'Luma bits' in Part B is the")
print(f"        effective Shannon entropy of the in-FOV luma channel (bits/px).")


Configuration OK  |  classes=['invasive cancer']  |  img=200x200
  sharpen=True (amount=0.8, ksize=5)
  FOV: gray_thr=15, open=5, close=11
  histogram matching:
    reference     = median
    match_space   = RGB
    strength      = 1.0  (1.0 = full match, 0.0 = identity)
    fov_only      = True
    clip          = True
  validation batch = 5 imgs/class
  NOTE: HM does NOT quantise, so 'Luma bits' in Part B is the
        effective Shannon entropy of the in-FOV luma channel (bits/px).


In [3]:
# ============================================================
# Cell 1.3 -- Sharpen + preprocess + load_dataset
#            (diverse synthetic fallback + SYNTHETIC_MODE flag)
# ============================================================
def sharpen(img_rgb, amount=None, ksize=None):
    """Unsharp-mask sharpening. Applied once after resize."""
    amount = SHARPEN_AMOUNT if amount is None else amount
    ksize  = SHARPEN_KSIZE  if ksize  is None else ksize
    if ksize % 2 == 0:
        ksize += 1                          # cv2.GaussianBlur requires odd ksize
    blurred = cv2.GaussianBlur(img_rgb, (ksize, ksize), 0)
    sharp   = cv2.addWeighted(img_rgb, 1.0 + amount, blurred, -amount, 0)
    return sharp.astype(np.uint8)


def preprocess_image(img_bgr, size=IMG_SIZE, sharpen_it=APPLY_SHARPEN):
    """BGR -> RGB -> resize -> (optional) sharpen. Single source of truth."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    if sharpen_it:
        img = sharpen(img)
    return img


def _find_lesion_masks(folder, stem, size):
    """Look up a clinician mask for the given image stem."""
    for mext in (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"):
        cand = os.path.join(folder, stem + mext)
        if os.path.isfile(cand):
            m = cv2.imread(cand, cv2.IMREAD_GRAYSCALE)
            if m is not None:
                m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
                return (m > 127).astype(bool), cand
    return None, None


def load_dataset(root, classes, size=IMG_SIZE, subset_per_class=None,
                 sharpen_it=APPLY_SHARPEN, masks_subdir="masks"):
    """Load images from <root>/<class>/images/ (or <root>/<class>/), resize,
    optionally sharpen. Case-insensitive dedupe by filename."""
    X, y, paths, masks = [], [], [], []
    n_with_mask_total = 0
    n_dedup_total = 0
    for cls in classes:
        folder = os.path.join(root, cls)
        if not os.path.isdir(folder):
            print(f"  [warn] folder not found: {folder}")
            continue
        img_folder = os.path.join(folder, "images")
        if not os.path.isdir(img_folder):
            img_folder = folder
        raw = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            raw.extend(glob.glob(os.path.join(img_folder, ext)))
        seen, files = set(), []
        n_dup_this = 0
        for p in sorted(raw):
            key = os.path.normpath(p).lower()
            if key in seen:
                n_dup_this += 1
                continue
            seen.add(key)
            files.append(p)
        n_dedup_total += n_dup_this
        if subset_per_class:
            files = files[:subset_per_class]
        mask_folder = os.path.join(folder, masks_subdir) if masks_subdir else None
        n_with_mask = 0
        for p in files:
            img = cv2.imread(p)
            if img is None:
                continue
            img = preprocess_image(img, size=size, sharpen_it=sharpen_it)
            stem = os.path.splitext(os.path.basename(p))[0]
            msk = None
            if mask_folder and os.path.isdir(mask_folder):
                msk, _ = _find_lesion_masks(mask_folder, stem, size)
                if msk is not None:
                    n_with_mask += 1
            if msk is None:
                msk = np.zeros((size, size), dtype=bool)
            X.append(img); y.append(cls); paths.append(p); masks.append(msk)
        n_with_mask_total += n_with_mask
        print(f"  {cls:>20}: {len(files):>4} images | masks matched: {n_with_mask}/{len(files)}  "
              f"| deduped: {n_dup_this}")
        if mask_folder and os.path.isdir(mask_folder) and n_with_mask == 0 and len(files) > 0:
            print(f"    [warn] masks/ folder found at {mask_folder} but 0/{len(files)} "
                  f"images matched. Check that mask filename stems match image stems.")
    if not X:
        empty = (np.zeros((0, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8),
                 np.array([]), [],
                 np.zeros((0, IMG_SIZE, IMG_SIZE), dtype=bool),
                 0, n_dedup_total)
        return empty
    return (np.array(X, dtype=np.uint8), np.array(y), paths,
            np.array(masks, dtype=bool), n_with_mask_total, n_dedup_total)


def make_synthetic_image_and_mask(H=200, W=200, seed=None):
    """Diverse synthetic 200x200 RGB histology-like image AND its lesion mask."""
    rng = np.random.RandomState(seed)
    grad_dir = rng.choice(['h', 'v', 'd1', 'd2'])
    base = rng.uniform(35, 80)
    slope = rng.uniform(15, 40)
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    if grad_dir == 'h':
        bg = base + slope * xx / W
    elif grad_dir == 'v':
        bg = base + slope * yy / H
    elif grad_dir == 'd1':
        bg = base + slope * (xx + yy) / (W + H)
    else:
        bg = base + slope * (xx - yy + W) / (2 * W)
    img = np.stack([bg, bg, bg], axis=-1)
    n_lesions = rng.randint(1, 4)
    yy2, xx2 = np.ogrid[:H, :W]
    mask = np.zeros((H, W), dtype=bool)
    for _ in range(n_lesions):
        cx = rng.randint(W // 5, 4 * W // 5)
        cy = rng.randint(H // 5, 4 * H // 5)
        r  = rng.randint(15, 50)
        disk = ((yy2 - cy) ** 2 + (xx2 - cx) ** 2 <= r ** 2)
        lr = rng.randint(180, 240)
        lg = rng.randint(120, 200)
        lb = rng.randint(60, 140)
        img[disk, 0] = lr; img[disk, 1] = lg; img[disk, 2] = lb
        mask |= disk
    texture = gaussian_filter(rng.uniform(-1, 1, (H, W)), sigma=8) * 20
    img = img + texture[..., None]
    noise = rng.randint(0, 15, (H, W, 3), dtype=np.uint8)
    return np.clip(img.astype(np.int32) + noise, 0, 255).astype(np.uint8), mask


# Load (or build synthetic fallback)
print("Loading dataset ...  (sharpen =", APPLY_SHARPEN, ")")
load_out = load_dataset(
    DATASET_ROOT, CLASSES,
    size=IMG_SIZE,
    subset_per_class=SUBSET_PER_CLASS,
)
X_raw, y_raw, paths_raw, masks_raw, n_with_mask, n_dedup = load_out
print(f"\nTotal images : {len(X_raw)} | masks matched: {n_with_mask}/{len(X_raw)} "
      f"| deduped (skipped duplicates): {n_dedup}")
if len(X_raw) > 0:
    SYNTHETIC_MODE = False
    print(f"Class counts : {dict(zip(*np.unique(y_raw, return_counts=True)))}")
    print(f"SYNTHETIC_MODE = {SYNTHETIC_MODE}")
else:
    SYNTHETIC_MODE = True
    print("\n*** No images found at:", DATASET_ROOT,
          "-- building DIVERSE synthetic fallback (10 imgs/class, with lesion masks).")
    X_synth, y_synth, m_synth = [], [], []
    for cls_idx, cls in enumerate(CLASSES):
        for k in range(10):
            img, msk = make_synthetic_image_and_mask(seed=SEED + cls_idx * 1000 + k)
            img = sharpen(img)
            X_synth.append(img); y_synth.append(cls); m_synth.append(msk)
    X_raw = np.stack(X_synth).astype(np.uint8)
    y_raw = np.array(y_synth)
    paths_raw = [f"<synthetic>/{c}/{i}.png" for c, i in zip(y_raw, range(len(y_raw)))]
    masks_raw = np.stack(m_synth).astype(bool)
    class_color_offset = {'invasive cancer': (25, 15, 10)}
    for i, cls in enumerate(y_raw):
        dr, dg, db = class_color_offset.get(cls, (0, 0, 0))
        X_raw[i] = np.clip(X_raw[i].astype(np.int32) + np.array([dr, dg, db]),
                           0, 255).astype(np.uint8)
    print(f"[fallback] Synthetic dataset ready: {len(X_raw)} images, "
          f"{int(masks_raw.any((1,2)).sum())}/{len(masks_raw)} with lesion masks "
          f"(SYNTHETIC_MODE=True).")


Loading dataset ...  (sharpen = True )
       invasive cancer:  128 images | masks matched: 0/128  | deduped: 0

Total images : 128 | masks matched: 0/128 | deduped (skipped duplicates): 0
Class counts : {np.str_('invasive cancer'): np.int64(128)}
SYNTHETIC_MODE = False


In [4]:
# ============================================================
# Cell 1.4 -- FOV mask + Histogram Matching preprocessing
# ============================================================
def compute_fov_mask(img_rgb, thr=FOV_GRAY_THR, open_k=FOV_OPEN_KSIZE,
                     close_k=FOV_CLOSE_KSIZE):
    """FOV mask: grey-threshold -> largest CC -> convex-hull fill -> close.
    Returns a (H, W) bool array. Identical definition to the rest of the
    suite so FOV-masked metrics are comparable across notebooks."""
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    m = (g > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,
                          np.ones((open_k, open_k), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1:
        return np.ones(g.shape, bool)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    fov = ndimage.binary_fill_holes(lab == k)
    ys, xs = np.where(fov)
    if len(ys) >= 3:
        pts = np.stack([xs, ys], axis=1).astype(np.int32)
        hull = cv2.convexHull(pts)
        hull_mask = np.zeros(g.shape, dtype=np.uint8)
        cv2.fillConvexPoly(hull_mask, hull, 1)
        hull_mask = hull_mask.astype(bool)
        x0, x1 = int(xs.min()), int(xs.max())
        y0, y1 = int(ys.min()), int(ys.max())
        bbox = np.zeros(g.shape, dtype=bool)
        bbox[y0:y1+1, x0:x1+1] = True
        hull_mask = hull_mask & bbox
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
        hull_mask = cv2.morphologyEx(hull_mask.astype(np.uint8),
                                      cv2.MORPH_CLOSE, kernel).astype(bool)
        fov = fov | hull_mask
    return fov


def build_reference_image(stack, fov_masks, mode=HM_REFERENCE):
    """Build the per-channel reference image for histogram matching.

    `mode`:
      "median" -- per-pixel median across the stack, in-FOV only
      "mean"   -- per-pixel mean across the stack, in-FOV only
      "first"  -- the first in-FOV image (deterministic, fast)
    Out-of-FOV pixels of the reference are set to 0 so match_histograms
    does not try to copy the black periphery.
    """
    N, H, W, _ = stack.shape
    ref = np.zeros((H, W, 3), dtype=np.float32)
    # union of all FOVs is the valid region of the reference
    union_fov = np.zeros((H, W), dtype=bool)
    for i in range(N):
        union_fov |= fov_masks[i]
    if not union_fov.any():
        # degenerate: take the whole frame
        union_fov = np.ones((H, W), dtype=bool)
    valid_idx = [i for i in range(N) if fov_masks[i].any()]
    if not valid_idx:
        valid_idx = list(range(N))
    if mode == "first":
        ref = stack[valid_idx[0]].astype(np.float32)
    else:
        # per-pixel reduce over the stack, ignoring out-of-FOV pixels
        acc = []
        for i in valid_idx:
            acc.append(stack[i].astype(np.float32))
        acc = np.stack(acc, axis=0)  # (M, H, W, 3)
        if mode == "median":
            ref = np.median(acc, axis=0)
        else:  # "mean"
            ref = acc.mean(axis=0)
    # zero out non-FOV region of the reference
    ref[~union_fov] = 0.0
    return np.clip(ref, 0, 255).astype(np.uint8)


def histogram_match_image(img_rgb, ref_rgb, fov_mask,
                          strength=HM_STRENGTH,
                          fov_only=HM_FOV_ONLY,
                          clip=HM_CLIP):
    """Match one image's per-channel histogram to the reference.

    Uses skimage.exposure.match_histograms (multi-channel, on RGB).
    Then linearly blends with the original: out = (1-s)*orig + s*matched.
    If `fov_only`, pixels outside `fov_mask` are forced to 0.
    """
    if not _SKIMAGE_OK:
        raise RuntimeError("scikit-image is required for histogram matching.")
    if strength <= 0.0:
        out = img_rgb.astype(np.uint8).copy()
    else:
        matched = _sk_match_hist(img_rgb, ref_rgb, channel_axis=-1)
        matched = np.clip(matched, 0, 255).astype(np.uint8)
        if strength >= 1.0:
            out = matched
        else:
            out = ((1.0 - strength) * img_rgb.astype(np.float32)
                   + strength * matched.astype(np.float32))
            out = np.clip(out, 0, 255).astype(np.uint8)
    if fov_only:
        out = out * fov_mask[..., None]
    if clip:
        out = np.clip(out, 0, 255).astype(np.uint8)
    return out


# ---- Compute FOV masks for every frame ----
print("Computing FOV masks ...")
fov_masks = np.stack([compute_fov_mask(X_raw[i]) for i in range(len(X_raw))])
print(f"  FOV masks ready: {fov_masks.shape}  "
      f"mean coverage = {fov_masks.mean():.3f}")

# ---- Build the reference image (frozen for the run) ----
print(f"\nBuilding reference image (mode = {HM_REFERENCE}) ...")
reference_image = build_reference_image(X_raw, fov_masks, mode=HM_REFERENCE)
print(f"  reference_image: shape={reference_image.shape} dtype={reference_image.dtype} "
      f"mean(in-FOV)={reference_image[fov_masks[0]].mean():.2f}")

# ---- Apply histogram matching to every frame ----
print(f"\nApplying histogram matching (strength={HM_STRENGTH}, "
      f"fov_only={HM_FOV_ONLY}) ...")
t0 = time.time()
X_hm = np.stack([
    histogram_match_image(X_raw[i], reference_image, fov_masks[i])
    for i in range(len(X_raw))
])
dt = time.time() - t0
print(f"  X_hm: shape={X_hm.shape} dtype={X_hm.dtype}  "
      f"mean per-image time = {1000*dt/max(len(X_raw),1):.1f} ms")

# ---- Sanity: identity check when strength=0 ----
if HM_STRENGTH == 0.0:
    assert np.array_equal(X_hm, X_raw * fov_masks[..., None]) or \
           np.array_equal(X_hm, X_raw), "HM with strength=0 should be identity"
    print("  [sanity] strength=0 -> output equals input (identity).")

# ---- Quick stats: how much did HM change the frames? ----
_diffs = np.abs(X_hm.astype(np.int32) - X_raw.astype(np.int32))
print(f"\nHM vs raw (in-FOV): mean |dRGB| = {_diffs[fov_masks].mean():.2f}  "
      f"max |dRGB| = {_diffs[fov_masks].max()}")

# Smoke test on the first image
_sm = histogram_match_image(X_raw[0], reference_image, fov_masks[0])
assert _sm.shape == X_raw[0].shape and _sm.dtype == np.uint8
assert np.array_equal(
    histogram_match_image(X_raw[0], reference_image, fov_masks[0]),
    _sm
), "histogram_match_image is not deterministic"
print(f"  [smoke] deterministic OK  |  out shape={_sm.shape} dtype={_sm.dtype}")


Computing FOV masks ...
  FOV masks ready: (128, 200, 200)  mean coverage = 0.957

Building reference image (mode = median) ...
  reference_image: shape=(200, 200, 3) dtype=uint8 mean(in-FOV)=167.46

Applying histogram matching (strength=1.0, fov_only=True) ...
  X_hm: shape=(128, 200, 200, 3) dtype=uint8  mean per-image time = 2.1 ms

HM vs raw (in-FOV): mean |dRGB| = 14.30  max |dRGB| = 127
  [smoke] deterministic OK  |  out shape=(200, 200, 3) dtype=uint8


# Bridge — hand off Histogram Matching preprocessing to Step 4

The validator (Part B) expects three variables from Part A:

* `X_base` — the **processed** stack, `(N, 200, 200, 3)` uint8 RGB. Here
  `X_base = X_hm` (the histogram-matched stack).
* `X_raw`  — the **resize-only** stack (no sharpen, no HM), used only as
  the source for the FOV mask so the mask is computed on unmodified
  pixels. Here we reload the dataset with `sharpen_it=False`.
* `masks`  — clinician lesion masks (synthetic in fallback), pass-through.

The former QI-SABP wrapper `qsaharp()` is **removed**. The only
preprocessing function exposed to Part B is `histogram_match_image()`,
called once above to build `X_hm`. Part B does not re-run any
preprocessing; it only measures `X_hm` against `X_raw`.


In [5]:
# ============================================================
# Bridge -- hand off Histogram Matching preprocessing to Step 4.
# ============================================================

# --- HM-processed stack becomes the validator's X_base ---
X_base = X_hm.copy()                        # histogram-matched, 200x200, uint8 RGB
_masks = masks_raw.copy()                   # clinician masks (synthetic in fallback)

# --- Unique image ids (always sequential -> guaranteed unique) ---
image_ids = np.array([f"img_{i:04d}" for i in range(len(X_base))],
                     dtype=object)
assert len(set(image_ids)) == len(X_base), \
    "image_ids not unique after bridge (this should never happen)"

# --- Reload the SAME images WITHOUT sharpening -> validator's X_raw ---
#     (used only as the source for the field-of-view mask)
if not SYNTHETIC_MODE:
    _X_unsharp, _y_unsharp, _paths_unsharp, _masks_unsharp, _n_wm, _n_dd = load_dataset(
        DATASET_ROOT, CLASSES,
        size=IMG_SIZE,
        subset_per_class=SUBSET_PER_CLASS,
        sharpen_it=False,                  # <-- resize-only
    )
    if len(_X_unsharp) == 0:               # safety net: fall back to X_raw
        X_raw_unsharp = X_raw.copy()
    else:
        X_raw_unsharp = _X_unsharp
        if len(X_raw_unsharp) != len(X_base):
            print(f"[bridge] X_raw_unsharp count ({len(X_raw_unsharp)}) != "
                  f"X_base count ({len(X_base)}); falling back to X_raw.")
            X_raw_unsharp = X_raw.copy()
else:
    # Synthetic path: rebuild the same synthetic images WITHOUT the
    # sharpen pass, so the validator gets a true resize-only FOV source.
    _xs = []
    for cls_idx, cls in enumerate(CLASSES):
        for k in range(10):
            img, _ = make_synthetic_image_and_mask(seed=SEED + cls_idx * 1000 + k)
            _xs.append(img)
    X_raw_unsharp = np.stack(_xs).astype(np.uint8)
    class_color_offset = {'invasive cancer': (25, 15, 10)}
    for i, cls in enumerate(y_raw):
        dr, dg, db = class_color_offset.get(cls, (0, 0, 0))
        X_raw_unsharp[i] = np.clip(
            X_raw_unsharp[i].astype(np.int32) + np.array([dr, dg, db]),
            0, 255,
        ).astype(np.uint8)

# Step 4 expects the name `X_raw` for the resize-only stack.
X_raw = X_raw_unsharp

# Final shape assertions
assert X_base.dtype == np.uint8 and X_base.ndim == 4 and X_base.shape[-1] == 3, \
    f"X_base must be (N,H,W,3) uint8, got {X_base.shape} {X_base.dtype}"
assert X_raw.shape == X_base.shape, \
    f"X_raw {X_raw.shape} != X_base {X_base.shape} after bridge"
assert len(image_ids) == len(X_base) == len(X_raw), \
    f"count mismatch: ids={len(image_ids)} X_base={len(X_base)} X_raw={len(X_raw)}"
assert _masks.shape == (len(X_base), IMG_SIZE, IMG_SIZE), \
    f"masks shape {_masks.shape} != ({len(X_base)}, {IMG_SIZE}, {IMG_SIZE})"

masks = _masks   # pass-through to Step 4
print(f"Bridge ready | X_base (HM) = {X_base.shape} {X_base.dtype} | "
      f"X_raw (resize-only) = {X_raw.shape} {X_raw.dtype} | "
      f"image_ids = {len(image_ids)} (all unique) | masks = {masks.shape} "
      f"({int(masks.any((1,2)).sum())}/{len(masks)} have a lesion) | "
      f"SYNTHETIC_MODE = {SYNTHETIC_MODE}")

# --- HM wrapper for Step 4 (no-op wrapper kept for API symmetry) ---
HM_DEFAULTS = {
    "reference": HM_REFERENCE,
    "strength":  HM_STRENGTH,
    "fov_only":  HM_FOV_ONLY,
    "clip":      HM_CLIP,
}

def hm_preprocess(img_uint8_rgb, params=None, seed=0):
    """Histogram-matching preprocessing, exposed to the Step 4 validator.

    Signature: hm_preprocess(img_uint8_RGB, params_dict, seed) -> uint8 RGB image.
    `params` and `seed` are accepted for API compatibility but ignored:
    the reference and strength are fixed at module level (no tuning on a
    single patient). The output of Part A is already in `X_hm`, so this
    wrapper simply returns `X_base[idx]` for the matching image; if the
    image is not in the stack, it matches on the fly against the frozen
    reference.
    """
    # find by exact match first (fast path)
    for i in range(len(X_base)):
        if np.array_equal(X_base[i], img_uint8_rgb):
            return X_base[i].copy()
    # slow path: re-match on the fly (used only by external callers)
    fov = compute_fov_mask(img_uint8_rgb)
    return histogram_match_image(img_uint8_rgb, reference_image, fov)

# Quick smoke test on the first image
_sm = hm_preprocess(X_base[0], HM_DEFAULTS, seed=12345)
assert _sm.shape == X_base[0].shape and _sm.dtype == np.uint8, \
    f"hm_preprocess smoke test failed: got {_sm.shape} {_sm.dtype}"
assert np.array_equal(hm_preprocess(X_base[0], HM_DEFAULTS, seed=12345), _sm), \
    "hm_preprocess is not deterministic for the same seed"
print(f"hm_preprocess() wrapper ready  |  HM_DEFAULTS = {HM_DEFAULTS}  |  "
      f"smoke test on X_base[0]: out shape={_sm.shape} dtype={_sm.dtype}  "
      f"mean|diff|={np.abs(_sm.astype(int)-X_base[0].astype(int)).mean():.3f}")


       invasive cancer:  128 images | masks matched: 0/128  | deduped: 0
Bridge ready | X_base (HM) = (128, 200, 200, 3) uint8 | X_raw (resize-only) = (128, 200, 200, 3) uint8 | image_ids = 128 (all unique) | masks = (128, 200, 200) (0/128 have a lesion) | SYNTHETIC_MODE = False
hm_preprocess() wrapper ready  |  HM_DEFAULTS = {'reference': 'median', 'strength': 1.0, 'fov_only': True, 'clip': True}  |  smoke test on X_base[0]: out shape=(200, 200, 3) dtype=uint8  mean|diff|=0.000


# Part B — Step 4 validation with 11 metrics

Single-patient, dHash clusters, all frames. The frames are NOT
independent. Descriptive only — no p-values, no robustness or population
claims.

**Technique:** Histogram Matching (HM) to a dataset-derived median
reference, applied per-frame in RGB.

**Metrics (11):** in 4 groups (fidelity, colour, diagnostic structure,
quality/cost). All computed inside the FOV mask only, per image, against
the ORIGINAL (resize-only) image.

| group | metric | direction |
|---|---|---|
| fidelity | PSNR (dB)               | up |
| fidelity | SSIM                    | up |
| colour   | dE76 (CIE76)            | down |
| colour   | a* correlation          | up |
| colour   | \|a* shift\|            | down |
| structure| Vessel Dice (top-10% Frangi) | up |
| structure| Edge IoU (Canny)        | up |
| quality  | Sharpness ratio (out/in)| neutral |
| quality  | Noise ratio (out/in)    | down |
| quality  | Saturation ratio (out/in)| neutral |
| cost     | Luma bits (effective Shannon entropy in-FOV, bits/px) | neutral |

**Tables only.** No figures, no plots. T1 (main), T5 (per cluster),
T6 (caveats). Identity is included as a reference row (ideal values,
not ranked).

**Note on `Luma bits`:** the former QI-SABP notebook reported an analytic
bit allocation (8 bits salient / 4 bits non-salient). That concept is
gone with QI-SABP. Histogram matching does not quantise, so the honest
replacement is the **effective Shannon entropy of the in-FOV luma
channel**, in bits/pixel. For an 8-bit image this lies in [0, 8]; it
measures how much of the 8-bit luma range is actually used. Identity
keeps the original entropy; HM usually raises it slightly because the
matching stretches the luma CDF to cover the reference range.


In [6]:
# ============================================================
# Step 4.1 -- Freeze inputs: load N frames, FOV masks, cluster ids.
#              (No cache_hash assert: the former QI-SABP Part-A-frozen
#               guard is not applicable to the HM pipeline. The hash is
#               printed for information only.)
# ============================================================
import hashlib as _hashlib
import numpy as np
import pandas as pd
import cv2
import time
from scipy import ndimage

# ---- dHash clustering (identical definition to the rest of the suite) ----
def _dhash_step4(gray):
    g = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    return (g[:, 1:] > g[:, :-1]).ravel()

def _compute_fov_step4(img_rgb, thr=15):
    """FOV mask (convex-hull filled, same as Part A's compute_fov_mask)."""
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    m = (g > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1:
        return np.ones(g.shape, bool)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    fov = ndimage.binary_fill_holes(lab == k)
    ys, xs = np.where(fov)
    if len(ys) >= 3:
        pts = np.stack([xs, ys], axis=1).astype(np.int32)
        hull = cv2.convexHull(pts)
        hull_mask = np.zeros(g.shape, dtype=np.uint8)
        cv2.fillConvexPoly(hull_mask, hull, 1)
        hull_mask = hull_mask.astype(bool)
        x0, x1 = int(xs.min()), int(xs.max())
        y0, y1 = int(ys.min()), int(ys.max())
        bbox = np.zeros(g.shape, dtype=bool)
        bbox[y0:y1+1, x0:x1+1] = True
        hull_mask = hull_mask & bbox
        ksize = 11
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
        hull_mask = cv2.morphologyEx(hull_mask.astype(np.uint8),
                                      cv2.MORPH_CLOSE, kernel).astype(bool)
        fov = fov | hull_mask
    return fov

# ---- Build the cache from X_base (Part A's histogram-matched stack) ----
images = X_base           # (N, 200, 200, 3) uint8 RGB, histogram-matched
images_raw = X_raw        # (N, 200, 200, 3) uint8 RGB, resize-only (for FOV)
n_images = len(images)
image_id = np.array([f"img_{i:04d}" for i in range(n_images)], dtype=object)
fov = np.stack([_compute_fov_step4(images_raw[i]) for i in range(n_images)])

# dHash clustering (same as the rest of the suite)
hs = [_dhash_step4(cv2.cvtColor(images[i], cv2.COLOR_RGB2GRAY)) for i in range(n_images)]
cluster_ids = np.full(n_images, -1, dtype=int)
_next = 0
for i, h in enumerate(hs):
    if cluster_ids[i] >= 0:
        continue
    cluster_ids[i] = _next
    for j in range(i + 1, n_images):
        if cluster_ids[j] < 0 and (h != hs[j]).sum() <= 4:
            cluster_ids[j] = _next
    _next += 1
n_clusters = len(np.unique(cluster_ids))
cluster_reps = np.array([np.where(cluster_ids == uc)[0][0]
                          for uc in np.unique(cluster_ids)], dtype=int)

# ---- cache_hash (informational only; NOT asserted) ----
# The former assert (cache_hash == 12f68b681210) was a QI-SABP Part-A
# frozen guard. The HM pipeline is a different preprocessing operator,
# so the guard no longer applies. We print the hash for traceability.
_h = _hashlib.md5("|".join(map(str, image_id)).encode())
_h.update(np.ascontiguousarray(images).tobytes())
cache_hash = _h.hexdigest()[:12]
print(f"cache_hash = {cache_hash}  (informational only; HM pipeline, not asserted)")
print(f"  n_images = {n_images} | n_clusters (dHash) = {n_clusters} | "
      f"n_distinct = {len(cluster_reps)}")
print(f"  The {n_images} frames are NOT independent ({n_clusters} distinct clusters "
      f"from one patient). All tables say this.")


cache_hash = 42d47682380f  (informational only; HM pipeline, not asserted)
  n_images = 128 | n_clusters (dHash) = 7 | n_distinct = 7
  The 128 frames are NOT independent (7 distinct clusters from one patient). All tables say this.


In [7]:
# ============================================================
# Step 4.2 -- step4_validator + helper functions.
#              Computes 11 metrics inside the FOV only.
# ============================================================
from skimage.metrics import structural_similarity as _ssim_fn
from skimage.color import deltaE_cie76 as _deltaE_fn, rgb2lab as _rgb2lab_fn

def _vessel_map(img_rgb):
    """Frangi vessel filter -> vessel probability map (0-1)."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    try:
        from skimage.filters import frangi
        v = frangi(gray, black_ridges=False)
        v = v / (v.max() + 1e-8)
    except Exception:
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        v = np.hypot(gx, gy)
        v = v / (v.max() + 1e-8)
    return v.astype(np.float32)

def _vessel_dice(original, processed, fov_mask, top_pct=10):
    """Dice of the top `top_pct` Frangi vessel pixels inside the FOV."""
    f = np.asarray(fov_mask, dtype=bool)
    v_o = _vessel_map(original)[f]
    v_p = _vessel_map(processed)[f]
    thr_o = np.percentile(v_o, 100 - top_pct)
    thr_p = np.percentile(v_p, 100 - top_pct)
    b_o = v_o >= thr_o
    b_p = v_p >= thr_p
    inter = float((b_o & b_p).sum())
    size_o = float(b_o.sum())
    size_p = float(b_p.sum())
    denom = size_o + size_p
    if denom == 0:
        return float('nan')
    return 2.0 * inter / denom

def _edge_map(img_rgb):
    """Canny edge binary map."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    return (edges > 0).astype(np.uint8)

def _pearson_corr(a, b, mask):
    """Pearson correlation inside the mask."""
    a = a[mask].astype(np.float64)
    b = b[mask].astype(np.float64)
    if len(a) < 2 or a.std() < 1e-9 or b.std() < 1e-9:
        return float('nan')
    return float(np.corrcoef(a, b)[0, 1])

def _edge_iou(a_edges, b_edges, mask):
    """IoU of two edge maps inside the mask."""
    a = a_edges[mask].astype(bool)
    b = b_edges[mask].astype(bool)
    u = (a | b).sum()
    if u == 0:
        return float('nan')
    return float((a & b).sum() / u)

def _specular_mask(img_rgb):
    """Specular/glare mask (V>=250 & S<=25 in HSV)."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    return ((hsv[..., 2] >= 250) & (hsv[..., 1] <= 25)).astype(bool)

def _luma_entropy_bits(img_rgb, mask, n_bins=256):
    """Effective Shannon entropy of the in-FOV luma channel, in bits/px.

    Replaces the former QI-SABP analytic bit allocation. HM does not
    quantise, so the honest 'Luma bits' reading is the information
    density actually present in the in-FOV luma channel. For an 8-bit
    image this lies in [0, 8]. A flat image -> 0; a uniform image -> 8.
    """
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    vals = g[mask].astype(np.int64)
    if vals.size == 0:
        return float('nan')
    hist = np.bincount(vals, minlength=n_bins).astype(np.float64)
    p = hist / hist.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def step4_validator(original, processed, fov_mask):
    """Compute 11 metrics inside the FOV only, per image.

    Returns a dict with keys m1..m11:
      m1  PSNR (dB)                         up
      m2  SSIM                              up
      m3  CIE76 dE                          down
      m4  a* correlation                     up
      m5  |mean a* shift|                    down
      m6  Vessel Dice (top-10% Frangi)       up
      m7  Edge IoU (Canny)                   up
      m8  Sharpness ratio (out/in)           neutral
      m9  Noise ratio (out/in)               down
      m10 Saturation ratio (out/in)          neutral
      m11 Luma bits (effective Shannon entropy, bits/px)  neutral
    """
    f = np.asarray(fov_mask, dtype=bool)
    o = np.asarray(original, dtype=np.float32)
    p = np.asarray(processed, dtype=np.float32)
    # m1: PSNR (dB)
    mse = float(((o - p) ** 2)[f].mean())
    m1 = 10.0 * np.log10(255.0 ** 2 / max(mse, 1e-10))
    # m2: SSIM (FOV-masked, mean over channels)
    ssim_map = _ssim_fn(original, processed, channel_axis=2,
                         full=True, data_range=255)[1]
    m2 = float(ssim_map.mean(2)[f].mean())
    # m3: CIE76 dE
    m3 = float(_deltaE_fn(_rgb2lab_fn(original), _rgb2lab_fn(processed))[f].mean())
    # m4: a* correlation
    lab_o = _rgb2lab_fn(original)
    lab_p = _rgb2lab_fn(processed)
    m4 = _pearson_corr(lab_o[..., 1], lab_p[..., 1], f)
    # m5: |mean a* shift|
    a_o = float(lab_o[..., 1][f].mean())
    a_p = float(lab_p[..., 1][f].mean())
    m5 = abs(a_p - a_o)
    # m6: Vessel Dice
    m6 = _vessel_dice(original, processed, f, top_pct=10)
    # m7: Edge IoU
    e_o = _edge_map(original)
    e_p = _edge_map(processed)
    m7 = _edge_iou(e_o, e_p, f)
    # m8: Sharpness ratio (Laplacian variance, out/in)
    g_o = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
    g_p = cv2.cvtColor(processed, cv2.COLOR_RGB2GRAY)
    lap_o = float(cv2.Laplacian(g_o, cv2.CV_64F)[f].var())
    lap_p = float(cv2.Laplacian(g_p, cv2.CV_64F)[f].var())
    m8 = lap_p / max(lap_o, 1e-9)
    # m9: Noise ratio (1.4826 * MAD of high-pass, in low-gradient regions)
    hp_o = g_o.astype(np.float32) - cv2.GaussianBlur(g_o.astype(np.float32), (0, 0), 1.5)
    hp_p = g_p.astype(np.float32) - cv2.GaussianBlur(g_p.astype(np.float32), (0, 0), 1.5)
    gm = np.hypot(cv2.Sobel(g_o.astype(np.float32), cv2.CV_32F, 1, 0),
                   cv2.Sobel(g_o.astype(np.float32), cv2.CV_32F, 0, 1))
    smooth = f & (gm < np.percentile(gm[f], 30))
    if smooth.any():
        noise_o = float(1.4826 * np.median(np.abs(hp_o[smooth] - np.median(hp_o[smooth]))))
        noise_p = float(1.4826 * np.median(np.abs(hp_p[smooth] - np.median(hp_p[smooth]))))
        m9 = noise_p / max(noise_o, 1e-9)
    else:
        m9 = float('nan')
    # m10: Saturation ratio (output specular / input specular)
    sp_o = _specular_mask(original)
    sp_p = _specular_mask(processed)
    m10 = float(sp_p[f].sum()) / max(float(sp_o[f].sum()), 1e-9)
    # m11: Luma bits (effective Shannon entropy of in-FOV luma, bits/px)
    m11 = _luma_entropy_bits(processed, f, n_bins=256)
    return dict(m1=m1, m2=m2, m3=m3, m4=m4, m5=m5, m6=m6, m7=m7,
                m8=m8, m9=m9, m10=m10, m11=m11)

# Metric metadata (key, name, group, better direction)
METRIC_META = [
    ("m1",  "PSNR (dB)",            "fidelity",  "up"),
    ("m2",  "SSIM",                 "fidelity",  "up"),
    ("m3",  "dE76",                 "colour",    "down"),
    ("m4",  "a* corr",              "colour",    "up"),
    ("m5",  "|a* shift|",          "colour",    "down"),
    ("m6",  "Vessel Dice",          "structure", "up"),
    ("m7",  "Edge IoU",             "structure", "up"),
    ("m8",  "Sharpness ratio",      "quality",   "neutral"),
    ("m9",  "Noise ratio",          "quality",   "down"),
    ("m10", "Saturation ratio",     "quality",   "neutral"),
    ("m11", "Luma bits",            "cost",      "neutral"),
]
METRIC_KEYS  = [m[0] for m in METRIC_META]
METRIC_NAMES = {m[0]: m[1] for m in METRIC_META}
METRIC_GROUPS= {m[0]: m[2] for m in METRIC_META}
METRIC_BETTER= {m[0]: m[3] for m in METRIC_META}
print(f"step4_validator ready: {len(METRIC_KEYS)} metrics (m1..m{len(METRIC_KEYS)}).")


step4_validator ready: 11 metrics (m1..m11).


In [8]:
# ============================================================
# Step 4.3 -- Compute 11 metrics for Histogram Matching.
#             Identity row (orig vs orig) is included as a reference.
#             Write per-image CSV. Cache processed images for T5.
# ============================================================
import os as _os
import glob as _glob

for _old in _glob.glob("results_*.csv"):
    _os.remove(_old)
    print(f"  Deleted stale {_old}")

TECHNIQUE = "HM"
processed_cache = {}   # (technique, image_idx) -> uint8 RGB

def _run_technique(name, processed_stack):
    """Compute 11 metrics for every image against its original.
    `processed_stack` is (N, H, W, 3) uint8 RGB."""
    rows = []
    for i in range(n_images):
        orig = images_raw[i]        # resize-only original (the reference)
        proc = processed_stack[i]   # the technique's output
        fov_i = fov[i]
        m = step4_validator(orig, proc, fov_i)
        row = {
            "technique": name,
            "image_id": image_id[i],
            "cluster_id": int(cluster_ids[i]),
        }
        row.update({k: m[k] for k in METRIC_KEYS})
        rows.append(row)
        processed_cache[(name, i)] = proc
    df = pd.DataFrame(rows)
    fname = f"results_{name}.csv"
    df.to_csv(fname, index=False)
    print(f"  wrote {fname}: {len(df)} rows")
    return df

# ---- HM ----
print("Computing 11 metrics for HM ...")
t0 = time.time()
df_hm = _run_technique("HM", X_base)
print(f"  HM done in {time.time()-t0:.1f}s")

# ---- Identity (reference row; orig vs orig) ----
print("Computing 11 metrics for Identity (reference) ...")
df_id = _run_technique("Identity", images_raw)
print(f"  Identity done.")

# ---- sanity: identity should be near-ideal ----
_id_psnr = df_id["m1"].mean()
_id_ssim = df_id["m2"].mean()
_id_de   = df_id["m3"].mean()
print(f"\n[identity sanity] PSNR={_id_psnr:.2f} dB | SSIM={_id_ssim:.4f} | dE76={_id_de:.4f}")
assert _id_psnr > 60.0, f"identity PSNR too low ({_id_psnr:.2f}); validator is broken"
assert _id_ssim > 0.999, f"identity SSIM too low ({_id_ssim:.4f}); validator is broken"
assert _id_de < 1e-3, f"identity dE76 too high ({_id_de:.4f}); validator is broken"
print("  identity row is near-ideal (PSNR>60, SSIM>0.999, dE<1e-3). OK.")


  Deleted stale results_Gabor_gain_0.25.csv
  Deleted stale results_GuidedFiltering.csv
Computing 11 metrics for HM ...
  wrote results_HM.csv: 128 rows
  HM done in 45.1s
Computing 11 metrics for Identity (reference) ...
  wrote results_Identity.csv: 128 rows
  Identity done.

[identity sanity] PSNR=148.13 dB | SSIM=1.0000 | dE76=0.0000
  identity row is near-ideal (PSNR>60, SSIM>0.999, dE<1e-3). OK.


In [9]:
# ============================================================
# Step 4.4 -- Merge cell: concatenate all technique CSVs. Assert
#              each technique has N rows and the same cluster ids.
#              Assert no NaNs (or list them).
# ============================================================

_csv_names = [
    "results_HM.csv",
    "results_Identity.csv",
]
print(f"Reading {len(_csv_names)} CSV files (fixed list, not glob)")
dfs = []
for f in _csv_names:
    if not _os.path.isfile(f):
        print(f"  [warn] {f} missing — skipping")
        continue
    _df = pd.read_csv(f)
    tech = _df["technique"].iloc[0] if len(_df) > 0 else f
    assert len(_df) == n_images, f"{f} has {len(_df)} rows, expected {n_images}"
    assert set(_df["image_id"]) == set(image_id), \
        f"{f} image_id set mismatch"
    assert list(_df["cluster_id"]) == list(cluster_ids), \
        f"{f} cluster_id order mismatch"
    dfs.append(_df)
    print(f"  {f}: {len(_df)} rows, technique={tech}")

all_df = pd.concat(dfs, ignore_index=True)
techniques = list(all_df["technique"].unique())
print(f"\nMerged: {len(all_df)} rows, techniques = {techniques}")

# NaN check
_nans = all_df[["technique"] + METRIC_KEYS].isna().sum()
_nans = _nans[_nans > 0]
if len(_nans):
    print("[NaNs found]")
    print(_nans)
else:
    print("No NaNs in any metric column. OK.")


Reading 2 CSV files (fixed list, not glob)
  [warn] results_HM.csv missing — skipping
  results_Identity.csv: 128 rows, technique=Identity

Merged: 128 rows, techniques = ['Identity']
No NaNs in any metric column. OK.


In [10]:
# ============================================================
# Step 4.5 -- Aggregation: average within each cluster first, then
#              across clusters. 95% CI from a cluster bootstrap.
#              Label: "within-patient, K clusters, not a population CI".
# ============================================================

def _cluster_mean(tech_df, metric_key):
    """Average within each cluster first, then across clusters."""
    cluster_means = tech_df.groupby("cluster_id")[metric_key].mean()
    return float(cluster_means.mean())

def _cluster_bootstrap_ci(tech_df, metric_key, n_boot=2000, seed=42):
    """Bootstrap CI over clusters (K clusters, 2000 resamples)."""
    cluster_means = tech_df.groupby("cluster_id")[metric_key].mean()
    uc = np.unique(tech_df["cluster_id"])
    if len(uc) < 2:
        m = float(cluster_means.mean()) if len(cluster_means) else float('nan')
        return m, float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(cluster_means), size=len(cluster_means))
        boots.append(cluster_means.iloc[idx].mean())
    boots = np.array(boots)
    lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
    return float(cluster_means.mean()), lo, hi

agg_rows = []
for tech in techniques:
    tech_df = all_df[all_df["technique"] == tech]
    row = {"technique": tech}
    for mk in METRIC_KEYS:
        mean, lo, hi = _cluster_bootstrap_ci(tech_df, mk)
        row[mk] = mean
        row[f"{mk}_lo"] = lo
        row[f"{mk}_hi"] = hi
    agg_rows.append(row)
agg_df = pd.DataFrame(agg_rows).set_index("technique")
print("Aggregation ready (cluster-weighted mean + 95% bootstrap CI).")
print(f"  techniques: {list(agg_df.index)}")
print(f"  clusters: {n_clusters}  |  frames: {n_images} (NOT independent)")


Aggregation ready (cluster-weighted mean + 95% bootstrap CI).
  techniques: ['Identity']
  clusters: 7  |  frames: 128 (NOT independent)


In [11]:
# ============================================================
# Step 4.6 -- Statistics: HM vs Identity, per key metric.
#              Uses paired difference per cluster (K values) + bootstrap
#              CI. Reports "CI excludes 0: yes/no".
#              Descriptive, no significance claim.
# ============================================================

def _paired_cluster_diff_ci(tech_df, ref_df, metric_key, n_boot=2000, seed=42):
    """Paired difference per cluster (tech - ref), bootstrap CI."""
    tech_cm = tech_df.groupby("cluster_id")[metric_key].mean()
    ref_cm  = ref_df.groupby("cluster_id")[metric_key].mean()
    diff = (tech_cm - ref_cm).values
    if len(diff) < 2:
        return float('nan'), float('nan'), float('nan'), False
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(diff), size=len(diff))
        boots.append(diff[idx].mean())
    boots = np.array(boots)
    lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
    excludes = (lo > 0) or (hi < 0)
    return float(diff.mean()), lo, hi, excludes

hm_df  = all_df[all_df["technique"] == "HM"].set_index("image_id")
id_df  = all_df[all_df["technique"] == "Identity"].set_index("image_id")

stat_rows = []
for mk in METRIC_KEYS:
    d, lo, hi, ex = _paired_cluster_diff_ci(hm_df, id_df, mk)
    stat_rows.append({
        "metric": METRIC_NAMES[mk],
        "key": mk,
        "HM_minus_Identity": d,
        "CI_lo": lo,
        "CI_hi": hi,
        "CI_excludes_0": "yes" if ex else "no",
        "better": METRIC_BETTER[mk],
    })
stat_df = pd.DataFrame(stat_rows)
print("Paired cluster-diff statistics ready (HM vs Identity). Descriptive only.")


Paired cluster-diff statistics ready (HM vs Identity). Descriptive only.


In [18]:
# ============================================================
# COMPLETE FIXED PIPELINE
# Merge + Cluster Aggregation + Bootstrap CI + Tables T1/T5/T6
#
# Techniques: TopHat and Identity
# Source folder: C:\CER
# Bootstrap: 2000 resamples, fixed seed
#
# T1 = Cluster-weighted means and 95% bootstrap CIs
# T5 = Per-cluster TopHat metrics
# T6 = Research caveats
#
# No glob. No dependence on stale notebook variables.
# ============================================================

from pathlib import Path
from collections import Counter
import re
import numpy as np
import pandas as pd

# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

RESULTS_DIR = Path(r"C:\CER")

SOURCE_FILES = {
    "TopHat": RESULTS_DIR / "results_TopHat.csv",
    "Identity": RESULTS_DIR / "results_Identity.csv",
}

OUTPUT_MERGED = RESULTS_DIR / "results_merged_tophat_identity.csv"
OUTPUT_T1 = RESULTS_DIR / "results_T1_TopHat_vs_Identity.csv"
OUTPUT_T5 = RESULTS_DIR / "results_T5_TopHat_per_cluster.csv"
OUTPUT_T6 = RESULTS_DIR / "results_T6_caveats.csv"
OUTPUT_AGG = RESULTS_DIR / "results_aggregation_TopHat_vs_Identity.csv"

TECHNIQUES = ["TopHat", "Identity"]
N_BOOT = 2000
BOOTSTRAP_SEED = 42
CLUSTER_TABLE_TECHNIQUE = "TopHat"

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

print("=" * 100)
print("TOPHAT + IDENTITY: COMPLETE ANALYSIS PIPELINE")
print("=" * 100)


# ============================================================
# STEP 2: CHECK SOURCE FILES
# ============================================================

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Results directory does not exist: {RESULTS_DIR}"
    )

missing_files = [
    str(path)
    for path in SOURCE_FILES.values()
    if not path.is_file()
]

if missing_files:
    available_csvs = sorted(
        p.name for p in RESULTS_DIR.iterdir()
        if p.is_file() and p.suffix.lower() == ".csv"
    )

    raise FileNotFoundError(
        "Required source CSV file(s) are missing:\n"
        + "\n".join(missing_files)
        + "\n\nCSV files currently available:\n"
        + "\n".join(available_csvs)
    )

print("\nSource files:")
for technique, path in SOURCE_FILES.items():
    print(f"  {technique}: {path.name}")


# ============================================================
# STEP 3: HELPER FUNCTIONS
# ============================================================

def normalize_name(value):
    """Normalize names by ignoring spaces, punctuation and case."""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def normalize_cluster_id(value):
    """Normalize numeric cluster IDs without changing the source IDs."""
    if pd.isna(value):
        raise ValueError("Missing cluster_id found.")

    value = str(value).strip()

    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = value.split(".")[0]

    if value == "":
        raise ValueError("Empty cluster_id found.")

    return value


# ============================================================
# STEP 4: READ AND VALIDATE THE TWO CSV FILES
# ============================================================

source_dfs = {}

for expected_technique, path in SOURCE_FILES.items():

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(f"{path.name} is empty.")

    required_columns = {"technique", "cluster_id"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    labels = (
        df["technique"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    if (
        len(labels) != 1
        or normalize_name(labels[0])
        != normalize_name(expected_technique)
    ):
        raise ValueError(
            f"Technique mismatch in {path.name}.\n"
            f"Expected: {expected_technique}\n"
            f"Found: {labels}"
        )

    df = df.copy()
    df["technique"] = expected_technique

    if df["cluster_id"].isna().any():
        raise ValueError(
            f"Missing cluster IDs in {path.name}."
        )

    df["_cluster_key"] = df["cluster_id"].map(
        normalize_cluster_id
    )

    source_dfs[expected_technique] = df

    print(
        f"LOADED: {path.name} | "
        f"Rows: {len(df)} | "
        f"Clusters: {df['_cluster_key'].nunique()}"
    )


# ============================================================
# STEP 5: VALIDATE CLUSTER MEMBERSHIP
# ============================================================

top_df = source_dfs["TopHat"]
identity_df = source_dfs["Identity"]

top_cluster_counts = (
    top_df.groupby("_cluster_key").size().sort_index().to_dict()
)

identity_cluster_counts = (
    identity_df.groupby("_cluster_key").size().sort_index().to_dict()
)

if top_cluster_counts != identity_cluster_counts:
    raise ValueError(
        "Cluster membership or frame counts differ between TopHat "
        "and Identity.\n"
        f"TopHat: {top_cluster_counts}\n"
        f"Identity: {identity_cluster_counts}\n"
        "Check the source CSV files before continuing."
    )

n_images = len(top_df)
n_clusters = top_df["_cluster_key"].nunique()

print(f"\nFrames per technique: {n_images}")
print(f"Matched clusters: {n_clusters}")
print("PASS: Both files have matching per-cluster frame counts.")


# ============================================================
# STEP 6: CHECK COMMON COLUMNS AND IDENTIFY METRICS
# ============================================================

top_original_columns = [
    col for col in top_df.columns if col != "_cluster_key"
]

identity_original_columns = [
    col for col in identity_df.columns if col != "_cluster_key"
]

top_column_set = set(top_original_columns)
identity_column_set = set(identity_original_columns)

common_columns = [
    col for col in top_original_columns
    if col in identity_column_set
]

if not {"technique", "cluster_id"}.issubset(common_columns):
    raise ValueError(
        "The two files do not share the required technique and "
        "cluster_id columns."
    )

top_only_columns = sorted(top_column_set - identity_column_set)
identity_only_columns = sorted(identity_column_set - top_column_set)

if top_only_columns or identity_only_columns:
    print("\nWARNING: Source schemas differ.")
    print("TopHat-only columns:", top_only_columns)
    print("Identity-only columns:", identity_only_columns)
    print("Only common columns will be retained in the comparison.")

excluded_normalized_names = {
    "technique",
    "clusterid",
    "cluster",
    "imageid",
    "imageindex",
    "imageidx",
    "imagename",
    "imagepath",
    "frameid",
    "frameidx",
    "frame",
    "patientid",
    "patient",
    "filename",
    "filepath",
    "path",
    "file",
    "index",
    "id",
    "rowid",
    "sampleid",
    "sampleindex",
    "nframes",
    "framecount",
    "unnamed0",
}

metric_keys = []

# Identify common numeric columns, ignoring ID and metadata columns.
for col in common_columns:

    if normalize_name(col) in excluded_normalized_names:
        continue

    if (
        pd.api.types.is_numeric_dtype(top_df[col])
        and pd.api.types.is_numeric_dtype(identity_df[col])
    ):
        metric_keys.append(col)

if not metric_keys:
    raise ValueError(
        "No common numeric metric columns were found. "
        "Inspect the source CSV headers and data types."
    )

# Convert metrics explicitly and reject missing/infinite values.
for technique in TECHNIQUES:

    df = source_dfs[technique]

    for metric in metric_keys:
        df[metric] = pd.to_numeric(df[metric], errors="raise")

    values = df[metric_keys].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        missing_counts = df[metric_keys].isna().sum()
        missing_counts = missing_counts[missing_counts > 0]

        raise ValueError(
            f"Invalid metric values in {technique}.\n"
            f"Missing values: {missing_counts.to_dict()}\n"
            "Check the source CSV before calculating the tables."
        )

print(f"\nCommon metric columns: {len(metric_keys)}")
for metric in metric_keys:
    print("  -", metric)


# ============================================================
# STEP 7: OPTIONAL FRAME-IDENTITY VALIDATION
# ============================================================

# If both files have a shared frame identifier, verify that the
# same frames are represented in each technique and cluster.

frame_id_candidates = {
    "imageid",
    "imageindex",
    "imageidx",
    "frameid",
    "frameidx",
    "filename",
    "filepath",
    "imagepath",
    "sourcefile",
}

frame_id_column = next(
    (
        col for col in common_columns
        if normalize_name(col) in frame_id_candidates
    ),
    None,
)

if frame_id_column is not None:

    if (
        top_df[frame_id_column].isna().any()
        or identity_df[frame_id_column].isna().any()
    ):
        print(
            f"WARNING: {frame_id_column} contains missing values. "
            "Exact frame-identity comparison was skipped."
        )
    else:
        top_frame_keys = Counter(
            zip(
                top_df["_cluster_key"],
                top_df[frame_id_column].astype(str),
            )
        )

        identity_frame_keys = Counter(
            zip(
                identity_df["_cluster_key"],
                identity_df[frame_id_column].astype(str),
            )
        )

        if top_frame_keys != identity_frame_keys:
            raise ValueError(
                "The frame identifiers or their cluster assignments "
                "differ between TopHat and Identity."
            )

        print(
            f"PASS: Matching frame identities verified using "
            f"'{frame_id_column}'."
        )

else:
    print(
        "\nNo common frame identifier column was found. "
        "Cluster frame counts were validated, but exact frame-by-frame "
        "identity could not be independently verified."
    )


# ============================================================
# STEP 8: CREATE AND SAVE THE MERGED CSV
# ============================================================

# Use only common columns to avoid manufacturing NaN values for
# columns that exist in only one source file.

merged_top = source_dfs["TopHat"][common_columns].copy()
merged_identity = source_dfs["Identity"][common_columns].copy()

merged = pd.concat(
    [merged_top, merged_identity],
    ignore_index=True,
)

merged.to_csv(OUTPUT_MERGED, index=False)

assert OUTPUT_MERGED.is_file()

print("\n" + "=" * 90)
print("STEP 4.4 COMPLETED SUCCESSFULLY")
print("=" * 90)
print("Merged file:", OUTPUT_MERGED)
print("Merged shape:", merged.shape)
print("\nRows per technique:")
print(merged.groupby("technique").size())


# ============================================================
# STEP 9: PREPARE THE ANALYSIS DATA
# ============================================================

analysis_df = merged.copy()

analysis_df["_cluster_key"] = analysis_df["cluster_id"].map(
    normalize_cluster_id
)

actual_techniques = set(analysis_df["technique"].unique())

if actual_techniques != set(TECHNIQUES):
    raise ValueError(
        f"Expected techniques {TECHNIQUES}; "
        f"found {sorted(actual_techniques)}."
    )

# Human-readable metric labels, when defined by the notebook.
if "METRIC_NAMES" in globals() and isinstance(METRIC_NAMES, dict):
    metric_names = {
        mk: METRIC_NAMES.get(mk, mk)
        for mk in metric_keys
    }
else:
    metric_names = {mk: mk for mk in metric_keys}


# ============================================================
# STEP 10: CLUSTER-WEIGHTED MEAN AND BOOTSTRAP CI
# ============================================================

def cluster_bootstrap_ci(
    tech_df,
    metric_key,
    n_boot=2000,
    seed=42,
):
    """
    Calculate the mean of cluster means with equal cluster weights.

    Bootstrap unit: cluster, not frame.
    Confidence interval: 2.5th and 97.5th percentiles.

    These intervals describe resampling of the observed clusters;
    they are not population-level confidence intervals.
    """

    cluster_means = (
        tech_df.groupby("_cluster_key", sort=True)[metric_key]
        .mean()
    )

    values = cluster_means.to_numpy(dtype=float)
    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    observed_mean = float(values.mean())

    if n < 2:
        return observed_mean, np.nan, np.nan

    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        low=0,
        high=n,
        size=(n_boot, n),
    )

    bootstrap_means = values[sampled_indices].mean(axis=1)

    ci_lo, ci_hi = np.percentile(
        bootstrap_means,
        [2.5, 97.5],
    )

    return (
        observed_mean,
        float(ci_lo),
        float(ci_hi),
    )


# ============================================================
# STEP 4.5: CALCULATE AGGREGATION FOR BOTH TECHNIQUES
# ============================================================

agg_rows = []

for technique in TECHNIQUES:

    tech_df = analysis_df[
        analysis_df["technique"] == technique
    ].copy()

    row = {
        "technique": technique,
        "n_frames": len(tech_df),
        "n_clusters": tech_df["_cluster_key"].nunique(),
    }

    for metric_index, metric in enumerate(metric_keys):

        mean, lo, hi = cluster_bootstrap_ci(
            tech_df=tech_df,
            metric_key=metric,
            n_boot=N_BOOT,
            seed=BOOTSTRAP_SEED + metric_index,
        )

        row[metric] = mean
        row[f"{metric}_lo"] = lo
        row[f"{metric}_hi"] = hi

    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows).set_index("technique")

print("\nSTEP 4.5 COMPLETED: cluster-weighted means and bootstrap CIs.")
print(f"Clusters: {n_clusters}")
print(f"Bootstrap resamples: {N_BOOT}")


# ============================================================
# STEP 4.7: T1 MAIN TABLE
# ============================================================

print("\n" + "=" * 110)
print("T1 MAIN: CLUSTER-WEIGHTED MEAN [95% CLUSTER-BOOTSTRAP CI]")
print("=" * 110)

print(f"Frames per technique: {n_images}")
print(f"Clusters: {n_clusters}")
print(
    "Frames are not independent observations when grouped within "
    "the same patient or source cluster."
)
print(
    "Identity is the reference condition, not a competing "
    "enhancement method."
)
print(
    "Intervals describe resampling of observed clusters; "
    "they are not population-level confidence intervals."
)
print("=" * 110)

t1_rows = []

for technique in TECHNIQUES:

    row = {"technique": technique}

    for metric in metric_keys:

        mean = agg_df.loc[technique, metric]
        lo = agg_df.loc[technique, f"{metric}_lo"]
        hi = agg_df.loc[technique, f"{metric}_hi"]

        if pd.isna(lo) or pd.isna(hi):
            formatted = f"{mean:.3f} [NA, NA]"
        else:
            formatted = f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"

        row[metric_names[metric]] = formatted

    t1_rows.append(row)

t1 = pd.DataFrame(t1_rows).set_index("technique")

print(t1.to_string())

t1.to_csv(OUTPUT_T1)


# ============================================================
# STEP 4.7: T5 PER-CLUSTER TABLE
# ============================================================

print("\n" + "=" * 110)
print(f"T5 PER CLUSTER: {CLUSTER_TABLE_TECHNIQUE} METRICS")
print("=" * 110)

cluster_df = analysis_df[
    analysis_df["technique"] == CLUSTER_TABLE_TECHNIQUE
].copy()

if cluster_df.empty:
    raise ValueError(
        f"No observations found for {CLUSTER_TABLE_TECHNIQUE}."
    )

t5_rows = []

for cluster_key, cluster_df_part in cluster_df.groupby(
    "_cluster_key",
    sort=True,
):

    row = {
        "cluster_id": cluster_df_part["cluster_id"].iloc[0],
        "n_frames": len(cluster_df_part),
    }

    for metric in metric_keys:
        row[metric_names[metric]] = float(
            cluster_df_part[metric].mean()
        )

    t5_rows.append(row)

t5 = pd.DataFrame(t5_rows).set_index("cluster_id")

print(t5.to_string(float_format=lambda value: f"{value:.3f}"))

t5.to_csv(OUTPUT_T5)


# ============================================================
# STEP 4.7: T6 CAVEATS
# ============================================================

print("\n" + "=" * 110)
print("T6 CAVEATS")
print("=" * 110)

caveats = [
    (
        f"The analysis contains {n_images} frames per technique "
        f"and {n_clusters} clusters. Frames from the same source "
        "should not be treated as independent patients."
    ),
    (
        "Identity is a reference condition (original versus original), "
        "not a competing enhancement method."
    ),
    (
        "PSNR, SSIM and color-difference metrics assess fidelity to "
        "the reference. A lower fidelity score alone does not establish "
        "that an enhanced image is clinically worse."
    ),
    (
        "Structural metrics depend on the segmentation masks, edge "
        "definitions and implementation used."
    ),
    (
        "Sharpness, noise and saturation ratios are descriptive. "
        "Their desirability depends on the image and intended application."
    ),
    (
        "The 95% percentile intervals resample observed clusters. "
        "They do not establish population-level uncertainty."
    ),
    (
        "This pipeline does not calculate p-values or make "
        "statistical-significance claims."
    ),
]

for i, caveat in enumerate(caveats, start=1):
    print(f"T6.{i}: {caveat}")

pd.DataFrame({"caveat": caveats}).to_csv(
    OUTPUT_T6,
    index=False,
)


# ============================================================
# STEP 11: SAVE NUMERIC AGGREGATION AND FINAL SUMMARY
# ============================================================

agg_df.to_csv(OUTPUT_AGG)

print("\n" + "=" * 100)
print("ALL STEPS COMPLETED SUCCESSFULLY")
print("=" * 100)

print("\nGenerated files:")
for path in [
    OUTPUT_MERGED,
    OUTPUT_T1,
    OUTPUT_T5,
    OUTPUT_T6,
    OUTPUT_AGG,
]:
    print(f"  {path}")

print("\nFinal validation:")
print("Techniques:", TECHNIQUES)
print("Frames per technique:", n_images)
print("Matched clusters:", n_clusters)
print("Metric columns:", len(metric_keys))
print("Merged data shape:", merged.shape)

print("\nRows per technique:")
print(merged.groupby("technique").size())

print("\nT1 preview:")
display(t1)

print("\nT5 preview:")
display(t5)

print("\nSTEP 4.4 + STEP 4.5 + STEP 4.7 FINISHED.")

TOPHAT + IDENTITY: COMPLETE ANALYSIS PIPELINE

Source files:
  TopHat: results_TopHat.csv
  Identity: results_Identity.csv
LOADED: results_TopHat.csv | Rows: 128 | Clusters: 7
LOADED: results_Identity.csv | Rows: 128 | Clusters: 7

Frames per technique: 128
Matched clusters: 7
PASS: Both files have matching per-cluster frame counts.

TopHat-only columns: []
Identity-only columns: ['sigma_n_Y']
Only common columns will be retained in the comparison.

Common metric columns: 11
  - m1
  - m2
  - m3
  - m4
  - m5
  - m6
  - m7
  - m8
  - m9
  - m10
  - m11
PASS: Matching frame identities verified using 'image_id'.

STEP 4.4 COMPLETED SUCCESSFULLY
Merged file: C:\CER\results_merged_tophat_identity.csv
Merged shape: (256, 14)

Rows per technique:
technique
Identity    128
TopHat      128
dtype: int64

STEP 4.5 COMPLETED: cluster-weighted means and bootstrap CIs.
Clusters: 7
Bootstrap resamples: 2000

T1 MAIN: CLUSTER-WEIGHTED MEAN [95% CLUSTER-BOOTSTRAP CI]
Frames per technique: 128
Clusters

,PSNR (dB),SSIM,dE76,a* corr,|a* shift|,Vessel Dice,Edge IoU,Sharpness ratio,Noise ratio,Saturation ratio,Luma bits
technique,,,,,,,,,,,
TopHat,"18.683 [18.332, 18.988]","0.752 [0.744, 0.760]","11.539 [11.012, 12.144]","0.892 [0.890, 0.894]","5.751 [5.519, 5.967]","0.803 [0.788, 0.813]","0.539 [0.513, 0.570]","3.185 [2.953, 3.421]","2.409 [2.365, 2.453]","2.972 [2.769, 3.209]","8.000 [8.000, 8.000]"
Identity,"148.131 [148.131, 148.131]","1.000 [1.000, 1.000]","0.000 [0.000, 0.000]","1.000 [1.000, 1.000]","0.000 [0.000, 0.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","7.637 [7.563, 7.703]"



T5 preview:


,n_frames,PSNR (dB),SSIM,dE76,a* corr,|a* shift|,Vessel Dice,Edge IoU,Sharpness ratio,Noise ratio,Saturation ratio,Luma bits
cluster_id,,,,,,,,,,,,
0,8,19.091380,0.763137,10.863169,0.893087,5.468323,0.806729,0.504844,2.799530,2.472028,2.913482,8.0
1,8,18.749937,0.754497,11.555454,0.886132,6.072918,0.814250,0.517481,3.175497,2.415433,2.638056,8.0
2,48,17.849149,0.738908,12.951816,0.891199,6.120697,0.761638,0.611271,3.521034,2.301444,3.467775,8.0
3,32,18.174546,0.734051,12.366065,0.894294,5.943657,0.811438,0.595226,3.687335,2.349766,3.429377,8.0
4,8,19.063858,0.761379,10.845200,0.893613,5.414971,0.805765,0.515068,2.888947,2.434979,2.865160,8.0
5,8,18.808801,0.754361,11.410217,0.890112,5.933148,0.820250,0.516664,3.321717,2.405055,2.636459,8.0
6,16,19.040600,0.760260,10.782104,0.894804,5.303688,0.803293,0.514223,2.898321,2.487308,2.853195,8.0



STEP 4.4 + STEP 4.5 + STEP 4.7 FINISHED.
